# 2.0 - Cleaning: Missing Data by Mechanism, and Leakage Prevention

**Question this notebook answers:** why was every missing value treated differently
depending on *why* it was missing, instead of one blanket imputation rule - and how was
leakage ruled out before a single model was trained?

This notebook loads the already-cleaned `train.parquet` and narrates decisions already
made and documented in `docs/cleaning_decisions.md` / `docs/FACTS.md` - it does not
re-run the cleaning notebook (`02_cleaning.ipynb`) or re-diagnose anything from the raw
CSV.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
%matplotlib inline

train = pd.read_parquet(Path('..') / 'data' / 'processed' / 'train.parquet')

n_nulls = train.isnull().sum().sum()
print(f'train.parquet: {train.shape}')
print(f'Total nulls remaining: {n_nulls}')


train.parquet: (172988, 89)
Total nulls remaining: 0


Zero nulls remain - not because the data started clean (several columns were over
80% missing), but because every null was resolved into one of four mechanisms, each with
a different treatment. Imputing everything the same way (e.g., "fill with the median")
would have destroyed signal in at least two of these mechanisms and inverted it in a
third.

## The four mechanisms (from `docs/FACTS.md` §4, not recomputed here)

In [2]:
mechanisms = pd.DataFrame([
    ('MNAR (informative absence)', '5.6% - 84.6%',
     'Null share stable across every vintage; default rate among nulls diverges from filled rows',
     'Binary flag + sentinel (999 preserves "higher=safer" ordering; -1 where none exists)'),
    ('Staged rollout - 2012 bureau block (~40 cols)', '7.0% - 10.05%',
     '100% null through 2011, ~52% in 2012, 0% from 2013 (column birth log)',
     'Sentinel -1, disambiguated by the era_pre_2012 flag'),
    ('Rollout + structural, stacked (6 cols)', '7.9% - 17.0%',
     'Falls sharply post-2012 but never reaches 0% - a structural residual persists',
     'Sentinel 999; one column (mths_since_recent_inq) also gets its own flag'),
    ('Sparse / informative (6 cols, 1,079 rows)', '0.0003% - 0.10%',
     'Affected rows default at 18.35% vs. 14.81% population rate',
     'Aggregate flag (sparse_bureau_missing) + per-column median imputation'),
    ('Structural, own mechanism (num_tl_120dpd_2m)', '13.13%',
     'Inverted default signal vs. the mths_since_* family (nulls default MORE, not less)',
     'Sentinel -1 (a count, no "higher=safer" ordering) + its own dedicated flag'),
], columns=['mechanism', 'pct_null_range', 'evidence', 'treatment'])
mechanisms


,mechanism,pct_null_range,evidence,treatment
0,MNAR (informative absence),5.6% - 84.6%,Null share stable across every vintage; defaul...,"Binary flag + sentinel (999 preserves ""higher=..."
1,Staged rollout - 2012 bureau block (~40 cols),7.0% - 10.05%,"100% null through 2011, ~52% in 2012, 0% from ...","Sentinel -1, disambiguated by the era_pre_2012..."
2,"Rollout + structural, stacked (6 cols)",7.9% - 17.0%,Falls sharply post-2012 but never reaches 0% -...,Sentinel 999; one column (mths_since_recent_in...
3,"Sparse / informative (6 cols, 1,079 rows)",0.0003% - 0.10%,Affected rows default at 18.35% vs. 14.81% pop...,Aggregate flag (sparse_bureau_missing) + per-c...
4,"Structural, own mechanism (num_tl_120dpd_2m)",13.13%,Inverted default signal vs. the mths_since_* f...,"Sentinel -1 (a count, no ""higher=safer"" orderi..."


## Seeing the evidence directly, on the loaded data

`era_pre_2012` and the `*_missing` flags are the disambiguation mechanism - they are
still in `train.parquet`, so their prevalence can be checked directly rather than taken
on faith:

In [3]:
flag_cols = ['era_pre_2012', 'emp_length_missing', 'mths_since_last_delinq_missing',
             'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']
flag_rates = (train[flag_cols].mean() * 100).round(2).rename('% of train rows flagged')
flag_rates.to_frame()


,% of train rows flagged
era_pre_2012,29.86
emp_length_missing,4.35
mths_since_last_delinq_missing,58.46
mths_since_recent_inq_missing,29.66
num_tl_120dpd_2m_missing,29.98
sparse_bureau_missing,0.49


`era_pre_2012` alone covers 29.86% of the training set - a direct consequence of a
methodological choice recorded in `docs/scope.md`: pre-2012 vintages were *kept* rather
than dropped, on the hypothesis that early-adopter borrowers might carry their own risk
profile. That choice is what makes the sentinel-and-flag treatment necessary in the first
place, and it is also what produces the largest train-vs-test distribution shift observed
later (`4.0`): the flag is ~30% of train and structurally 0% of validation/test/2015 data.

## Leakage, screened on three fronts (temporal front covered in `4.0`)

**Target leakage.** Of the 151 raw columns, 37 are *post-origination* - written into the
record after the loan's outcome was already known - and were dropped before any modeling,
confirmed empirically via univariate AUC (not by name heuristics alone):
`recoveries` (AUC 0.90), `collection_recovery_fee` (AUC 0.86), `last_fico_range_low`
(AUC 0.91 after inverting direction). A column that alone nearly determines the outcome
was written *after* the outcome - this is the leakage test working as intended, not a
surprising result. None of these 37 columns exist in `train.parquet`; this is a "what was
excluded and why" narration, not a live check.

**Identity leakage.** `member_id`, the only borrower identifier in the raw file, is
100% null across the entire 2007-2018 history - confirmed in `docs/column_inventory.csv`.
Group-aware splitting (`GroupKFold`, keeping one borrower entirely on one side of a split)
is the textbook-correct defense against a repeat borrower appearing in both train and
test, and it is **not applicable here**: there is no key to group by. This is declared as
a known, unresolved limitation in `docs/scope.md`, not silently ignored.

In [4]:
assert 'member_id' not in train.columns
assert 'recoveries' not in train.columns and 'collection_recovery_fee' not in train.columns
print('Confirmed: no post-origination columns and no borrower identifier reached train.parquet.')


Confirmed: no post-origination columns and no borrower identifier reached train.parquet.


## Takeaway

Missing data here was never a single "how much is missing" question - it was "why is it
missing, and does the absence itself carry information?" Two mechanisms (MNAR, the sparse
bureau block) had the *null itself* correlate with risk; treating them as random missing
data and imputing blindly would have erased the strongest signal in those columns. The
`era_pre_2012` flag this produced is not a nuisance variable - it is the largest source of
train/test distribution shift in the whole project, and its own predictive value is
checked directly in `6.0`.

**Next:** `3.0-features.ipynb` - the five engineered ratio features that survived, and the
two that didn't.